In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# PREMIUM COLAB CELL: ВИБІР ФІЧ + ПОВНОВАЖКИЙ ТРЕНІНГ ОПТИМАЛЬНИХ МОДЕЛЕЙ
# ==============================================================================
# 
# УНІФІКОВАНА АРХІТЕКТУРА КОНСЕНСУСУ:
# - Всі моделі (легкі + важкі) тренують ОДНАКОВІ таргети
# - Легкі моделі: швидкі, ефективні (20-60 фіч)
# - Важкі моделі: глибокі, складні (80-120 фіч)
# - Порівняння: найкраща легка vs найкраща важка → консенсус
# 
# АВТОМАТИЧНА СИНХРОНІЗАЦІЯ ПАРАМЕТРІВ:
# - Локально: python run_hybrid_pipeline.py --test-ticker AMD --epochs 5
# - Параметри автоматично зберігаються в runtime_params.json
# - Colab автоматично читає і використовує ці параметри
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. ПІДКЛЮЧЕННЯ ШЛЯХІВ
# ------------------------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/trading_project'
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, PROJECT_PATH)
    print(f"✅ Google Drive підключено")
except ImportError:
    PROJECT_PATH = str(Path.cwd())
    if PROJECT_PATH not in sys.path:
        sys.path.insert(0, PROJECT_PATH)
    print("⚠️ Працюємо локально")

BATCH_NAME = "main_database"
batch_dir = Path(f"data/colab/accumulated/{BATCH_NAME}")
models_dir = batch_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 2. АВТОМАТИЧНЕ ЗАВАНТАЖЕННЯ ПАРАМЕТРІВ З runtime_params.json
# ==============================================================================

# ✅ FIX: Спочатку шукаємо runtime_params.json щоб дізнатися batch_name
runtime_params_path_src = Path("src/config/runtime_params.json")
if runtime_params_path_src.exists():
    with open(runtime_params_path_src, 'r') as f:
        runtime_params_temp = json.load(f)
    BATCH_NAME = runtime_params_temp.get('batch', {}).get('batch_name', 'main_database')
    print(f"📦 Batch name з runtime_params.json: {BATCH_NAME}")
    batch_dir = Path(f"data/colab/accumulated/{BATCH_NAME}")
    models_dir = batch_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
else:
    print(f"⚠️ runtime_params.json не знайдено, використовуємо за замовчуванням: {BATCH_NAME}")

runtime_params_path = batch_dir / "runtime_params.json"
print(f"\n🔍 DEBUG: Шукаємо runtime_params.json в {runtime_params_path}")
print(f"🔍 DEBUG: Файл існує? {runtime_params_path.exists()}")

# ✅ FIX: Якщо файл не знайдено в batch_dir, шукаємо в батьківській директорії
if not runtime_params_path.exists():
    alt_path = batch_dir.parent / "runtime_params.json"
    print(f"🔍 DEBUG: Шукаємо альтернативний шлях: {alt_path}")
    if alt_path.exists():
        print(f"✅ Знайдено в альтернативному шляху, копіюємо...")
        import shutil
        shutil.copy2(alt_path, runtime_params_path)
        runtime_params_path = alt_path

if runtime_params_path.exists():
    with open(runtime_params_path, 'r') as f:
        runtime_params = json.load(f)
    
    test_mode = runtime_params.get('test_mode', {})
    TEST_TICKER = test_mode.get('test_ticker')
    TEST_TARGET = test_mode.get('test_target')
    REDUCED_EPOCHS = test_mode.get('reduced_epochs')
    MAX_ITERATIONS = runtime_params.get('models', {}).get('max_iterations', 100)
    
    print("\n" + "="*80)
    print("📥 ПАРАМЕТРИ ЗАВАНТАЖЕНО З runtime_params.json")
    print("="*80)
    print(f"  Режим: {runtime_params.get('mode', 'full')}")
    tickers_list = runtime_params.get('training', {}).get('tickers', [])
    print(f"  Тікери: {tickers_list if tickers_list else 'всі з конфігу'}")
    timeframes_list = runtime_params.get('training', {}).get('timeframes', ['15m', '1h', '1d'])
    print(f"  Таймфрейми: {timeframes_list}")
    if TEST_TICKER:
        print(f"  🧪 Тестовий тікер: {TEST_TICKER}")
    if TEST_TARGET:
        print(f"  🧪 Тестовий таргет: {TEST_TARGET}")
    if REDUCED_EPOCHS:
        print(f"  ⚡ Епохи: {REDUCED_EPOCHS} (замість 50)")
    if MAX_ITERATIONS != 100:
        print(f"  ⚡ Ітерації: {MAX_ITERATIONS} (замість 100)")
    print("="*80 + "\n")
else:
    # Fallback до параметрів за замовчуванням
    TEST_TICKER = None
    TEST_TARGET = None
    REDUCED_EPOCHS = None
    MAX_ITERATIONS = 100
    print("\n⚠️ runtime_params.json не знайдено, використовуємо параметри за замовчуванням\n")

# ==============================================================================
# 3. СЛУЖБОВІ ФУНКЦІЇ ДЛЯ КЕШУ ТА МЕТРИК
# ==============================================================================
def compute_data_signature(df_feat, df_targ):
    feat_info = f"{df_feat.shape}_{pd.util.hash_pandas_object(df_feat.tail(100)).sum()}"
    targ_info = f"{df_targ.shape}_{pd.util.hash_pandas_object(df_targ.tail(100)).sum()}"
    combined = f"{feat_info}_{targ_info}"
    return hashlib.md5(combined.encode()).hexdigest()

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else 0.0
    return {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2), 'mape': float(mape)}

# ==============================================================================
# 4. АРХІТЕКТУРИ ВАЖКИХ МОДЕЛЕЙ (Повна реалізація)
# ==============================================================================
def create_model(model_type, input_size):
    if model_type == 'mlp':
        return nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    elif model_type == 'lstm':
        class LSTMModel(nn.Module):
            def __init__(self, input_sz):
                super().__init__()
                self.lstm = nn.LSTM(input_sz, 64, 2, batch_first=True, dropout=0.2)
                self.fc = nn.Linear(64, 1)
            def forward(self, x):
                out, _ = self.lstm(x.unsqueeze(1))
                return self.fc(out[:, -1, :])
        return LSTMModel(input_size)
    elif model_type == 'gru':
        class GRUModel(nn.Module):
            def __init__(self, input_sz):
                super().__init__()
                self.gru = nn.GRU(input_sz, 64, 2, batch_first=True, dropout=0.2)
                self.fc = nn.Linear(64, 1)
            def forward(self, x):
                out, _ = self.gru(x.unsqueeze(1))
                return self.fc(out[:, -1, :])
        return GRUModel(input_size)
    elif model_type == 'cnn':
        class CNNModel(nn.Module):
            def __init__(self, input_sz):
                super().__init__()
                self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
                self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
                self.pool = nn.AdaptiveAvgPool1d(1)
                self.fc = nn.Linear(64, 1)
            def forward(self, x):
                x = x.unsqueeze(1)
                x = torch.relu(self.conv1(x))
                x = torch.relu(self.conv2(x))
                return self.fc(self.pool(x).squeeze(-1))
        return CNNModel(input_size)
    elif model_type == 'transformer':
        class TransformerModel(nn.Module):
            def __init__(self, input_sz):
                super().__init__()
                self.embedding = nn.Linear(input_sz, 64)
                encoder_layer = nn.TransformerEncoderLayer(64, 4, dim_feedforward=128, dropout=0.2, batch_first=True)
                self.transformer = nn.TransformerEncoder(encoder_layer, 2)
                self.fc = nn.Linear(64, 1)
            def forward(self, x):
                x = self.embedding(x.unsqueeze(1))
                x = self.transformer(x)
                return self.fc(x[:, -1, :])
        return TransformerModel(input_size)
    elif model_type == 'tabnet':
        # Fallback для TabNet у Colab щоб уникнути конфліктів залежностей (використовуємо потужний MLP)
        return nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    elif model_type == 'autoencoder':
        class AutoencoderModel(nn.Module):
            def __init__(self, input_sz):
                super().__init__()
                self.encoder = nn.Sequential(
                    nn.Linear(input_sz, 64), nn.ReLU(),
                    nn.Linear(64, 32), nn.ReLU()
                )
                self.decoder = nn.Sequential(
                    nn.Linear(32, 16), nn.ReLU(),
                    nn.Linear(16, 1)
                )
            def forward(self, x):
                return self.decoder(self.encoder(x))
        return AutoencoderModel(input_size)
    else:
        # Fallback для будь-якої іншої моделі
        return nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1)
        )

# ==============================================================================
# 5. ЗАВАНТАЖЕННЯ ДАНИХ ТА ПЕРЕВІРКА КЕШУ
# ==============================================================================
print(f"📥 Завантаження бази {BATCH_NAME}...")
if not (batch_dir / "features.parquet").exists() or not (batch_dir / "targets.parquet").exists():
    raise FileNotFoundError(f"Файли відсутні в {batch_dir}")

features_df = pd.read_parquet(batch_dir / "features.parquet")
targets_df = pd.read_parquet(batch_dir / "targets.parquet")

print(f"✅ Базу завантажено: Features {features_df.shape}, Targets {targets_df.shape}")

# ==============================================================================
# 5.0 ІНІЦІАЛІЗАЦІЯ КОНФІГ-МЕНЕДЖЕРА
# ==============================================================================
from src.config.unified_config_manager import UnifiedConfigManager
from src.features.selection.smart_selector import SmartFeatureSelector

config_manager = UnifiedConfigManager(config_dir=str(Path(PROJECT_PATH) / "src/config"))

# ==============================================================================
# 5.1 РОЗРАХУНОК CONTEXT MAP (для важких моделей)
# ==============================================================================
print("\n🧪 Розрахунок Context Map для важких моделей...")
try:
    from src.features.enrichers.context_map_enricher import ContextMapEnricher
    
    context_enricher = ContextMapEnricher(config=config_manager.get_config('features') or {})
    
    # Розраховуємо context_map для кожного тікера/таймфрейму
    # Для спрощення: розраховуємо на всіх даних разом
    features_df = context_enricher.enrich(features_df, ticker='any', timeframe='any')
    
    print(f"✅ Context Map розраховано. Додано колонки: context_map, context_dow, context_hour, context_fingerprint")
    print(f"   Features shape після: {features_df.shape}")
except Exception as e:
    print(f"⚠️ Помилка розрахунку Context Map: {e}")
    print("   Продовжуємо без Context Map")

current_sig = compute_data_signature(features_df, targets_df)
cache_file = batch_dir / "colab_cache_sig.json"
force_recalculate = False

if cache_file.exists() and not force_recalculate:
    with open(cache_file, 'r') as f:
        cached_data = json.load(f)
    if cached_data.get('signature') == current_sig:
        print("\n✅ ДАНІ НЕ ЗМІНИЛИСЯ. Пропускаємо глобальний перерахунок (доучуємо залишки).")
    else:
        print("\n⚠️ Дані змінилися! Глобальний кеш скинуто.")
else:
    print("\n🆕 Нова сесія. Кеш бази відсутній.")

# ==============================================================================
# 6. ДИНАМІЧНЕ ЗАВАНТАЖЕННЯ НАЛАШТУВАНЬ ТА МОДЕЛЕЙ
# ==============================================================================
feature_selector = SmartFeatureSelector(
    config_manager=config_manager,
    storage_path="src/config/selected_features_cache.json"
)

# Читаємо моделі напряму з models.yaml через ConfigManager
models_config = config_manager.get('models', {})
if hasattr(models_config, 'as_dict'):
    models_config = models_config.as_dict()

cat = models_config.get('categories', {})
per_mod = models_config.get('per_model', {})

light_models = cat.get('light', ['catboost', 'lightgbm', 'xgboost', 'random_forest', 'linear', 'svm', 'knn', 'ensemble'])
heavy_models = cat.get('heavy', ['mlp', 'cnn', 'lstm', 'gru', 'transformer', 'tabnet', 'autoencoder'])

# ✅ УНІФІКОВАНА АРХІТЕКТУРА: всі моделі тренують однакові таргети
# Різниця тільки в кількості фіч та складності моделі
print(f"\n📊 Знайдено моделей:")
print(f"  💡 Легкі ({len(light_models)}): {', '.join(light_models)}")
print(f"  🔥 Важкі ({len(heavy_models)}): {', '.join(heavy_models)}")
print(f"\n💡 Легкі моделі: швидкі, ефективні (20-60 фіч)")
print(f"🔥 Важкі моделі: глибокі, складні (80-120 фіч)")
print(f"🎯 Консенсус: найкраща легка vs найкраща важка → сигнал\n")

tickers = [t for t in targets_df['ticker'].unique() if t]

# 🧪 Фільтруємо тікери якщо встановлено TEST_TICKER
if TEST_TICKER:
    if TEST_TICKER in tickers:
        tickers = [TEST_TICKER]
        print(f"🧪 Фільтровано тікери: {tickers}")
    else:
        print(f"⚠️ Тікер {TEST_TICKER} не знайдено. Використовуємо всі: {tickers}")

# ==============================================================================
# 7. ГОЛОВНИЙ ЦИКЛ (TICKERS -> TARGETS -> HEAVY MODELS)
# ==============================================================================
# Тренуємо ТІЛЬКИ важкі моделі в Colab (легкі тренуються локально)
# Для кожного тікера + таргету вибираємо оптимальні фічі та тренуємо
metadata_cols = ['ticker', 'timeframe', 'interval', 'datetime', 'date', 'hash', 'symbol']

for ticker in tickers:
    ticker_file = batch_dir / f"colab_results_{ticker}.json"
    
    # [GRANULAR RESUME] Спробуємо завантажити існуючі результати для тікера
    ticker_json = {
        "ticker": ticker,
        "timestamp": datetime.now().isoformat(),
        "total_trained": 0,
        "total_failed": 0,
        "timeframes": {}
    }
    
    if ticker_file.exists() and not force_recalculate:
        try:
            with open(ticker_file, 'r') as f:
                loaded_json = json.load(f)
                # Перевіряємо чи це той самий тікер
                if loaded_json.get('ticker') == ticker:
                    ticker_json = loaded_json
                    print(f"\n📂 Знайдено існуючі результати для {ticker}. Вмикаємо гранулярне доучування.")
        except Exception as e:
            print(f"⚠️ Не вдалося завантажити існуючий файл для {ticker}: {e}")

    print(f"\n{'='*80}\n🚀 ОБРОБКА ТІКЕРА: {ticker}\n{'='*80}")
    
    t_feat = features_df[features_df['ticker'] == ticker]
    t_targ = targets_df[targets_df['ticker'] == ticker]
    
    if t_feat.empty or t_targ.empty:
        print("  ⚠️ Даних немає, пропускаю.")
        continue
    
    # ✅ FIX: Merge на ['ticker', 'interval'] може створити дублікати
    # Потрібно також фільтрувати по таймфрейму
    # Спочатку отримуємо унікальні таймфрейми з таргетів
    available_timeframes = t_targ['interval'].unique() if 'interval' in t_targ.columns else ['1d']
    
    print(f"  📊 Features: {t_feat.shape}, Targets: {t_targ.shape}")
    print(f"  ⏱️ Таймфрейми: {available_timeframes}")
    
    # ✅ FIX: Merge має включати datetime для уникнення декартового добутку
    # КРИТИЧНО: Без datetime merge створює картезіанський добуток (1.5M рядків)
    # З datetime + inner join отримуємо правильне вирівнювання (~13k рядків)
    common_cols = ['ticker', 'interval']
    if 'datetime' in t_feat.columns and 'datetime' in t_targ.columns:
        common_cols.append('datetime')
        print(f"  ✅ Використовуємо datetime для merge (КРИТИЧНО для правильного вирівнювання)")
    else:
        print(f"  ⚠️ УВАГА: datetime не знайдено в обох датафреймах!")
        print(f"     Features columns: {list(t_feat.columns)[:10]}")
        print(f"     Targets columns: {list(t_targ.columns)[:10]}")
    
    # ВАЖЛИВО: inner join для уникнення дублікатів
    merged = pd.merge(t_feat, t_targ, on=common_cols, how='inner')
    print(f"  ✅ Merged: {merged.shape} (inner join з {len(common_cols)} ключами)")
    
    if merged.empty:
        print("  ⚠️ Merge результат пустий, пропускаю.")
        continue
    
    target_cols = [c for c in merged.columns if c.startswith('target_')]
    
    # 🧪 Фільтруємо таргети якщо встановлено TEST_TARGET
    if TEST_TARGET:
        if TEST_TARGET in target_cols:
            target_cols = [TEST_TARGET]
            print(f"🧪 Фільтровано таргети: {target_cols}")
        else:
            print(f"⚠️ Таргет {TEST_TARGET} не знайдено. Використовуємо всі: {len(target_cols)} таргетів")
    
    for target_col in target_cols:
        tf = target_col.split('_')[-1]
        print(f"\n  🎯 Таргет: {target_col}")
        
        if tf not in ticker_json["timeframes"]:
            ticker_json["timeframes"][tf] = {"trained": 0, "failed": 0, "results": {}}
            
        if target_col not in ticker_json["timeframes"][tf]["results"]:
            ticker_json["timeframes"][tf]["results"][target_col] = {"models": {}}
            
        mask = merged[target_col].notna()
        if mask.sum() < 50:
            print(f"    ⚠️ Лише {mask.sum()} зразків, занадто мало.")
            continue
        
        # ✅ DEBUG: Log data size before feature selection
        print(f"    📊 Data size: {mask.sum()} samples, {len(merged.columns)} columns")
        
        # ✅ FIX: Якщо дані занадто великі, зменшуємо для вибору фіч
        # SmartSelector може вичерпати пам'ять на великих датасетах
        max_samples_for_selection = 50000  # Максимум 50k зразків для вибору фіч
        if mask.sum() > max_samples_for_selection:
            print(f"    ⚠️ Дані занадто великі ({mask.sum()} > {max_samples_for_selection}), використовуємо вибірку")
            sample_idx = np.random.choice(np.where(mask)[0], size=max_samples_for_selection, replace=False)
            X_sample = merged.iloc[sample_idx].drop(columns=[c for c in metadata_cols if c in merged.columns] + target_cols, errors='ignore')
            y_sample = merged.iloc[sample_idx][target_col].fillna(0)
        else:
            X_sample = merged.loc[mask].drop(columns=[c for c in metadata_cols if c in merged.columns] + target_cols, errors='ignore')
            y_sample = merged.loc[mask, target_col].fillna(0)
            
        # Формування чистого X для вибору фіч
        X_df = X_sample
        y_ser = y_sample
        
        # Видаляємо дати та обробляємо типи
        X_df = X_df.select_dtypes(exclude=['datetime64', 'datetime', 'datetimetz'])
        for b_col in X_df.select_dtypes(include=['bool']).columns:
            X_df[b_col] = X_df[b_col].astype(float)
        for col in X_df.select_dtypes(include=['object']).columns:
            if X_df[col].nunique() < 20: 
                dummies = pd.get_dummies(X_df[col], prefix=col, drop_first=True)
                X_df = pd.concat([X_df.drop(columns=[col]), dummies], axis=1)
            else: X_df = X_df.drop(columns=[col], errors='ignore')
        X_df = X_df.apply(pd.to_numeric, errors='coerce').fillna(0).replace([np.inf, -np.inf], 0)
        
        all_feature_names = X_df.columns.tolist()

        # ✅ Тренуємо ТІЛЬКИ важкі моделі (легкі тренуються локально)
        for m_type in heavy_models:
            # [GRANULAR SKIP CHECK] Перевіряємо чи ця модель вже навчена
            m_res_node = ticker_json["timeframes"][tf]["results"][target_col]["models"]
            if m_type in m_res_node and m_res_node[m_type].get("trained", False):
                print(f"    ⏭️ {m_type:<14} | Вже навчена. Пропускаємо.")
                continue

            print(f"    🔍 {m_type:<14} | ", end="")
            
            try:
                # 7.1. ВИБІР ФІЧ ДЛЯ КОЖНОЇ МОДЕЛІ ОКРЕМО
                # ✅ Кожна модель отримує свій набір фіч (з models.yaml per_model)
                # Легкі: 20-60 фіч, Важкі: 80-120 фіч
                max_feats = per_mod.get(m_type, {}).get('max_features', 100)
                
                selected_features = feature_selector.select(
                    X=X_df, y=y_ser, context_id=f"{ticker}_{target_col}_{m_type}",
                    task_type="regression", model_type=m_type, return_metadata=False,
                    max_features=max_feats
                )
                
                selected_set = set(selected_features)
                last_row = X_df.iloc[-1]
                
                context_in = {}
                context_out = {}
                for f in all_feature_names:
                    val = float(last_row[f]) if not pd.isna(last_row[f]) else 0.0
                    if f in selected_set: context_in[f] = val
                    else: context_out[f] = val
                
                # 📋 Вивід контекстної карти для видимості
                in_features_preview = ', '.join(list(context_in.keys())[:3])
                out_features_preview = ', '.join(list(context_out.keys())[:3])
                if len(context_in) > 3:
                    in_features_preview += f", ... (+{len(context_in)-3})"
                if len(context_out) > 3:
                    out_features_preview += f", ... (+{len(context_out)-3})"
                
                res = {
                    "selected_features": selected_features,
                    "feature_count": len(selected_features),
                    "feature_selection_info": {"вибрані": context_in, "невибрані": context_out},  # Метаінформація про вибір фіч
                    "trained": False,
                    "mse": 0.0,
                    "model_path": ""
                }
                
                # 7.1. ТРЕНУВАННЯ ВАЖКИХ МОДЕЛЕЙ (MINI-BATCH, EARLY STOPPING)
                if len(selected_features) > 0:
                    available_features = [f for f in selected_features if f in X_df.columns]
                    
                    # --- TURBO OPTIMIZATION (Phase 7+) ---
                    # Deduplicate training data (X + y) before split to save 99% of computation time
                    # if redundant data is detected.
                    combined_data = pd.DataFrame(X_df[available_features])
                    combined_data['__target__'] = y_ser.values
                    
                    original_len = len(combined_data)
                    unique_data = combined_data.drop_duplicates()
                    unique_len = len(unique_data)
                    
                    if unique_len < original_len:
                        print(f"⚡ Turbo: {original_len} -> {unique_len} unique samples ({100*(1-unique_len/original_len):.1f}% reduction) | ", end="")
                        X_vals = unique_data.drop(columns=['__target__']).values
                        y_vals = unique_data['__target__'].values
                    else:
                        X_vals = X_df[available_features].values
                        y_vals = y_ser.values

                    X_tr, X_va, y_tr, y_va = train_test_split(X_vals, y_vals, test_size=0.2, random_state=42, shuffle=True)
                    
                    scaler = StandardScaler()
                    X_tr_sc = scaler.fit_transform(X_tr)
                    X_va_sc = scaler.transform(X_va)
                    
                    X_tr_t = torch.FloatTensor(X_tr_sc)
                    y_tr_t = torch.FloatTensor(y_tr).reshape(-1, 1)
                    X_va_t = torch.FloatTensor(X_va_sc)
                    y_va_t = torch.FloatTensor(y_va).reshape(-1, 1)
                    
                    model = create_model(m_type, len(available_features))
                    criterion = nn.MSELoss()
                    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
                    
                    # 🧪 Використовуємо REDUCED_EPOCHS якщо встановлено
                    epochs = REDUCED_EPOCHS if REDUCED_EPOCHS else 50
                    batch_size = 32
                    patience, patience_counter = 10, 0
                    best_loss = float('inf')
                    best_model_state = None
                    
                    for ep in range(epochs):
                        model.train()
                        for i in range(0, len(X_tr_t), batch_size):
                            batch_X = X_tr_t[i:i+batch_size]
                            batch_y = y_tr_t[i:i+batch_size]
                            
                            optimizer.zero_grad()
                            outputs = model(batch_X)
                            loss = criterion(outputs, batch_y)
                            loss.backward()
                            optimizer.step()
                            
                        # Валідація
                        model.eval()
                        with torch.no_grad():
                            val_outputs = model(X_va_t)
                            val_loss = criterion(val_outputs, y_va_t).item()
                            
                        if val_loss < best_loss:
                            best_loss = val_loss
                            patience_counter = 0
                            best_model_state = model.state_dict()
                        else:
                            patience_counter += 1
                            if patience_counter >= patience:
                                break
                    
                    # Зберігаємо кращу модель
                    if best_model_state:
                        model.load_state_dict(best_model_state)
                        
                    model.eval()
                    with torch.no_grad():
                        y_pred_tr = model(X_tr_t).numpy().flatten()
                        y_pred_va = model(X_va_t).numpy().flatten()
                        
                    tr_met = compute_metrics(y_tr, y_pred_tr)
                    va_met = compute_metrics(y_va, y_pred_va)
                    
                    m_path = models_dir / f"{m_type}_{ticker}_{target_col}.pt"
                    # ✅ FIX: Зберігаємо з повною інформацією для правильного завантаження
                    torch.save({
                        'model_state_dict': model.state_dict(),
                        'model_type': m_type,
                        'input_size': len(available_features),
                        'scaler': scaler,
                        'features': available_features
                    }, m_path)
                    
                    res.update({
                        "trained": True,
                        "model_path": str(m_path),
                        "mse": best_loss,
                        "best_loss": best_loss,
                        "train_metrics": tr_met,
                        "test_metrics": va_met
                    })
                    
                    ticker_json["timeframes"][tf]["trained"] += 1
                    ticker_json["total_trained"] += 1
                    print(f"✅ OK (MSE: {best_loss:.5f}, R²: {va_met['r2']:.3f})")
                
                # Додаємо результати у відповідний вузол
                k = ticker_json["timeframes"][tf]["results"][target_col]
                # Формат як очікує colab accumulator
                if "models" not in k: k["models"] = {}
                k["models"][m_type] = res
                
            except Exception as e:
                print(f"Помилка: {str(e)[:100]}")
                ticker_json["timeframes"][tf]["failed"] += 1
                ticker_json["total_failed"] += 1
                
                k = ticker_json["timeframes"][tf]["results"][target_col]
                if "models" not in k: k["models"] = {}
                k["models"][m_type] = {"error": str(e), "trained": False}

    # ЗБЕРЕЖЕННЯ Ticker Batch
    with open(ticker_file, 'w') as f:
        json.dump(ticker_json, f, indent=2)
    print(f"💾 Результати {ticker} збережено")

# ЗБЕРЕЖЕННЯ ГЛОБАЛЬНОГО ХЕШУ ТА СИНХРОНІЗАЦІЯ
with open(cache_file, 'w') as f:
    json.dump({'signature': current_sig, 'date': datetime.now().isoformat()}, f)

print("\n" + "="*80)
print("🎉 ВСІ ВАЖКІ МОДЕЛІ СФОРМОВАНІ!")
print(f"👉 Можна вантажити colab_results_*.json назад локально.")
print("="*80)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive підключено
📥 Завантаження бази main_database...
✅ Базу завантажено: Features (2026, 135), Targets (4120, 23)

✅ ДАНІ НЕ ЗМІНИЛИСЯ. Пропускаємо глобальний перерахунок (доучуємо залишки).
2026-04-07 07:02:36,734 - ProjectLogger - INFO - Logging configured. Level: DEBUG. Path: logs
2026-04-07 07:02:36,738 - src.core.security.secure_secrets_manager - DEBUG - Завантаження змінних оточення з файлу .env...
2026-04-07 07:02:36,742 - src.core.security.secure_secrets_manager - INFO - Змінні з .env (20 шт.) успішно завантажені в оточення.
2026-04-07 07:02:37,282 - ResourceMonitor - INFO - ResourceMonitor ініціалізовано з порогами: {'cpu_warning': 70.0, 'cpu_critical': 90.0, 'memory_warning': 80.0, 'memory_critical': 95.0, 'disk_warning': 85.0, 'disk_critical': 95.0}
2026-04-07 07:02:37,812 - UnifiedConfigManager - INFO - Loading configurations from: /cont